In [ ]:
import torch
import torch.nn as nn
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import json

# 1. 配置LoRA参数
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # 因果语言模型任务，可以选择不同类型的任务
    inference_mode=False,          # 训练模式
    r=8,                          # LoRA秩
    lora_alpha=32,                # LoRA alpha参数
    lora_dropout=0.1,             # LoRA dropout
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]  # 目标模块
)

# 2. 加载模型和tokenizer
model_name = "microsoft/DialoGPT-medium"  # 可以替换为其他模型
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 添加pad_token如果不存在
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 应用LoRA配置到模型
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # 打印可训练参数数量

# 3. 准备数据（示例数据）
def prepare_dataset():
    # 示例对话数据
    conversations = [
        {"input": "你好，今天天气怎么样？", "response": "今天天气很好，阳光明媚。"},
        {"input": "你能帮我写一封邮件吗？", "response": "当然可以，请告诉我邮件的内容。"},
        {"input": "什么是人工智能？", "response": "人工智能是计算机科学的一个分支，旨在创造能够执行智能任务的机器。"},
        # 添加更多数据...
    ]
    
    # 格式化数据
    formatted_data = []
    for conv in conversations:
        text = f"用户: {conv['input']}\n助手: {conv['response']}{tokenizer.eos_token}"
        formatted_data.append({"text": text})
    
    return Dataset.from_list(formatted_data)

dataset = prepare_dataset()

# 4. 数据预处理函数
def tokenize_function(examples):
    # 对文本进行tokenize
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=512,
        return_tensors=None
    )
    
    # 对于因果语言模型，标签就是输入本身
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_function,
    remove_columns=dataset.column_names,
    batched=True
)

# 5. 设置训练参数
training_args = TrainingArguments(
    output_dir="./lora_finetuned_model",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=100,
    learning_rate=2e-4,
    logging_steps=50,
    save_steps=500,
    evaluation_strategy="no",
    save_total_limit=2,
    prediction_loss_only=True,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),  # 如果GPU支持，使用混合精度训练
)

# 6. 创建数据收集器
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # 不使用掩码语言模型
)

# 7. 创建Trainer并开始训练
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("开始训练...")
trainer.train()

# 8. 保存LoRA适配器
trainer.save_model()
tokenizer.save_pretrained("./lora_finetuned_model")

print("训练完毕")

# 9. 推理示例
def generate_response(model, tokenizer, input_text, max_length=100):
    model.eval()
    
    # 格式化输入
    prompt = f"用户: {input_text}\n助手:"
    
    inputs = tokenizer(prompt, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 提取助手的回复
    assistant_response = response.split("助手:")[-1].strip()
    
    return assistant_response

# 测试推理
test_input = "你好，能介绍一下你自己吗？"
response = generate_response(model, tokenizer, test_input)
print(f"用户: {test_input}")
print(f"助手: {response}")